# Stage 03 — Replication

Reproduce the paper's main OLS/IV regressions. Write JSON + LaTeX table.
Follow `skills/stage_03.md` for detailed instructions.

In [1]:
from paths import *
import json
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm

df   = pd.read_parquet(str(DATASET_PATH))
spec = json.loads(PAPER_SPEC.read_text())

outcome   = spec['identification']['outcome_variable']
treatment = spec['identification']['treatment_variable']
instrument = spec['identification'].get('instrument')
controls  = spec['identification']['controls']
id_type   = spec['identification']['type']

print(f'Identification: {id_type}')
print(f'N obs (complete): {df[[outcome, treatment]].dropna().shape[0]}')

Identification: OLS
N obs (complete): 85


In [2]:
# --- Run 12 OLS regressions: 4 outcomes x 3 treatments, all 12 controls, HC1 robust SE ---

outcomes = [
    {"variable": "Investment2005", "label": "Investment as % of GDP (Panel A)"},
    {"variable": "FDI2005",        "label": "FDI as % of GDP (Panel B)"},
    {"variable": "nbusinessdensitycf", "label": "Business density (Panel C)"},
    {"variable": "nentryrateav",   "label": "Entry rate (Panel D)"},
]

treatments_list = [
    {"variable": "statutory",     "label": "Statutory rate"},
    {"variable": "effective",     "label": "First-year effective rate"},
    {"variable": "effective_5yr", "label": "Five-year effective rate"},
]

# Reference OLS results from Baiardi & Naghi Table 1 Column 8 (all controls)
# Only available for effective_5yr treatment
reference_ols = {
    ("Investment2005",        "effective_5yr"): {"coef": -0.189, "se": 0.118, "n": 61},
    ("FDI2005",               "effective_5yr"): {"coef": -0.095, "se": 0.081, "n": 61},
    ("nbusinessdensitycf",    "effective_5yr"): {"coef": -0.070, "se": 0.103, "n": 60},
    ("nentryrateav",          "effective_5yr"): {"coef": -0.133, "se": 0.103, "n": 50},
}

replication_specs = []

for out_info in outcomes:
    y_var = out_info["variable"]
    for tr_info in treatments_list:
        t_var = tr_info["variable"]
        
        # Build regression variables
        all_vars = [y_var, t_var] + controls
        sub = df[all_vars].dropna()
        n_obs = len(sub)
        
        if n_obs < len(controls) + 2:
            print(f"SKIP {y_var} ~ {t_var}: only {n_obs} obs")
            continue
        
        # OLS with HC1 robust SE
        X = sm.add_constant(sub[[t_var] + controls])
        y = sub[y_var]
        model = sm.OLS(y, X).fit(cov_type='HC1')
        
        coef_val = float(model.params[t_var])
        se_val = float(model.bse[t_var])
        ci = model.conf_int().loc[t_var]
        ci_lo = float(ci[0])
        ci_hi = float(ci[1])
        pval = float(model.pvalues[t_var])
        r2 = float(model.rsquared)
        
        # Check if we have a reference value
        ref_key = (y_var, t_var)
        has_ref = ref_key in reference_ols
        
        entry = {
            "spec": f"{out_info['label']} ~ {tr_info['label']}, all 12 controls",
            "table": f"Table 5D — {out_info['label']}",
            "outcome": y_var,
            "treatment": t_var,
            "replicated_coef": coef_val,
            "replicated_se": se_val,
            "ci_lo": ci_lo,
            "ci_hi": ci_hi,
            "p_value": pval,
            "r_squared": r2,
            "n_obs": n_obs,
            "se_type": "HC1",
            "n_controls": len(controls),
        }
        
        if has_ref:
            ref = reference_ols[ref_key]
            entry["published_coef"] = ref["coef"]
            entry["published_se"] = ref["se"]
            entry["published_n"] = ref["n"]
            entry["abs_diff"] = abs(coef_val - ref["coef"])
        
        replication_specs.append(entry)
        
        sig = "***" if pval < 0.01 else "**" if pval < 0.05 else "*" if pval < 0.1 else ""
        ref_str = f"  (ref: {reference_ols[ref_key]['coef']:.3f}, N_ref={reference_ols[ref_key]['n']})" if has_ref else ""
        print(f"{y_var:25s} ~ {t_var:15s}  coef={coef_val:8.4f}  se={se_val:.4f}  p={pval:.3f}{sig}  N={n_obs}{ref_str}")

print(f"\nTotal regressions: {len(replication_specs)}")
print(f"Controls ({len(controls)}): {', '.join(controls)}")

Investment2005            ~ statutory        coef= -0.0645  se=0.1209  p=0.594  N=61
Investment2005            ~ effective        coef= -0.1165  se=0.1123  p=0.299  N=61
Investment2005            ~ effective_5yr    coef= -0.1887  se=0.1157  p=0.103  N=61  (ref: -0.189, N_ref=61)
FDI2005                   ~ statutory        coef= -0.0300  se=0.0781  p=0.701  N=61
FDI2005                   ~ effective        coef= -0.1004  se=0.0601  p=0.094*  N=61
FDI2005                   ~ effective_5yr    coef= -0.0953  se=0.0716  p=0.183  N=61  (ref: -0.095, N_ref=61)
nbusinessdensitycf        ~ statutory        coef= -0.0344  se=0.0832  p=0.680  N=60
nbusinessdensitycf        ~ effective        coef= -0.0685  se=0.0948  p=0.470  N=60
nbusinessdensitycf        ~ effective_5yr    coef= -0.0695  se=0.0995  p=0.485  N=60  (ref: -0.070, N_ref=60)
nentryrateav              ~ statutory        coef= -0.0291  se=0.0721  p=0.686  N=50
nentryrateav              ~ effective        coef= -0.0834  se=0.1010  p=0

In [3]:
# --- Compare to published results (effective_5yr regressions only) ---
# Compute relative differences for specs that have reference values
# With correct controls, sample sizes should match reference exactly

THRESHOLD_PCT = 5.0  # Standard replication threshold per stage_03.md

for r in replication_specs:
    if "published_coef" in r:
        pub = r["published_coef"]
        rep = r["replicated_coef"]
        rdiff = abs(rep - pub) / abs(pub) * 100 if pub != 0 else float("nan")
        r["rel_diff_pct"] = round(rdiff, 2)
        r["match"] = rdiff < THRESHOLD_PCT
        r["threshold_pct"] = THRESHOLD_PCT
        
        n_match = "Y" if r["n_obs"] == r["published_n"] else "N"
        print(f"{r['spec']:65s}  pub={pub:.4f}  rep={rep:.4f}  diff={rdiff:.1f}%  N_match={n_match}  {'PASS' if r['match'] else 'FAIL'}")
    else:
        # No reference available — mark as not compared
        r["rel_diff_pct"] = None
        r["match"] = None
        r["threshold_pct"] = None
        print(f"{r['spec']:65s}  rep={r['replicated_coef']:.4f}  (no reference)")

# Summary
compared = [r for r in replication_specs if r.get("match") is not None]
n_pass = sum(1 for r in compared if r["match"])
print(f"\nReplication summary: {n_pass}/{len(compared)} within {THRESHOLD_PCT}% threshold")

Investment as % of GDP (Panel A) ~ Statutory rate, all 12 controls  rep=-0.0645  (no reference)
Investment as % of GDP (Panel A) ~ First-year effective rate, all 12 controls  rep=-0.1165  (no reference)
Investment as % of GDP (Panel A) ~ Five-year effective rate, all 12 controls  pub=-0.1890  rep=-0.1887  diff=0.1%  N_match=Y  PASS
FDI as % of GDP (Panel B) ~ Statutory rate, all 12 controls        rep=-0.0300  (no reference)
FDI as % of GDP (Panel B) ~ First-year effective rate, all 12 controls  rep=-0.1004  (no reference)
FDI as % of GDP (Panel B) ~ Five-year effective rate, all 12 controls  pub=-0.0950  rep=-0.0953  diff=0.4%  N_match=Y  PASS
Business density (Panel C) ~ Statutory rate, all 12 controls       rep=-0.0344  (no reference)
Business density (Panel C) ~ First-year effective rate, all 12 controls  rep=-0.0685  (no reference)
Business density (Panel C) ~ Five-year effective rate, all 12 controls  pub=-0.0700  rep=-0.0695  diff=0.7%  N_match=Y  PASS
Entry rate (Panel D) ~ Sta

In [4]:
# --- Write replication_results.json and replication_check.json ---

# Only count specs with reference values for pass/fail
compared_specs = [r for r in replication_specs if r.get("match") is not None]
n_pass = sum(1 for r in compared_specs if r["match"])
n_compared = len(compared_specs)
overall_pass = n_pass == n_compared if n_compared > 0 else True

replication_results = {
    "specs": replication_specs,
    "overall_pass": overall_pass,
    "n_specs": len(replication_specs),
    "n_compared": n_compared,
    "n_pass": n_pass,
    "notes": (
        "12 OLS regressions (4 outcomes x 3 treatments) with all 12 controls and HC1 robust SE. "
        "Reference values available only for effective_5yr treatment from Baiardi & Naghi (2024) Table 1 Col 8. "
        f"Replication threshold: {THRESHOLD_PCT}% relative difference."
    )
}
REPLICATION_RESULTS.write_text(json.dumps(replication_results, indent=2))

worst = max((r["rel_diff_pct"] for r in compared_specs), default=0)
replication_check = {
    "pass": overall_pass,
    "summary": f"{n_pass}/{n_compared} reference specs replicated within {THRESHOLD_PCT}% ({len(replication_specs)} total regressions run)",
    "worst_rel_diff_pct": worst,
    "failure_notes": None if overall_pass else (
        f"Some replications exceed the {THRESHOLD_PCT}% relative difference threshold. "
        "See individual spec entries for details."
    )
}
REPLICATION_CHECK.write_text(json.dumps(replication_check, indent=2))

print(f"Replication results: {n_pass}/{n_compared} matched (threshold {THRESHOLD_PCT}%)")
print(f"Worst relative difference: {worst:.1f}%")
print(f"Overall pass: {overall_pass}")
print(f"\n=> {REPLICATION_RESULTS}")
print(f"=> {REPLICATION_CHECK}")

Replication results: 4/4 matched (threshold 5.0%)
Worst relative difference: 0.7%
Overall pass: True

=> C:\Users\qgallea\Dropbox\work\claude_code\recast\papers\04_djankov_et_al_taxes\data\results\replication_results.json
=> C:\Users\qgallea\Dropbox\work\claude_code\recast\papers\04_djankov_et_al_taxes\data\results\replication_check.json


In [5]:
# --- Write LaTeX replication table ---
# Table shows all 12 regressions organized by outcome (panels) and treatment (columns)

treatments_order = ["statutory", "effective", "effective_5yr"]
treatment_labels = {"statutory": "Statutory", "effective": "Effective (1yr)", "effective_5yr": "Effective (5yr)"}
outcome_panels = [
    ("Investment2005", "Panel A: Investment/GDP"),
    ("FDI2005", "Panel B: FDI/GDP"),
    ("nbusinessdensitycf", "Panel C: Business Density"),
    ("nentryrateav", "Panel D: Entry Rate"),
]

# Build lookup
results_lookup = {}
for r in replication_specs:
    results_lookup[(r["outcome"], r["treatment"])] = r

lines = []
lines.append(r"\begin{table}[htbp]")
lines.append(r"\centering")
lines.append(r"\caption{Replication of OLS Regressions --- All 12 Controls, HC1 Robust SE}")
lines.append(r"\label{tab:replication}")
lines.append(r"\small")
lines.append(r"\begin{tabular}{l" + "c" * len(treatments_order) + "}")
lines.append(r"\hline\hline")

# Column headers
header = " & ".join([" "] + [treatment_labels[t] for t in treatments_order]) + r" \\"
lines.append(header)
lines.append(r"\hline")

for y_var, panel_label in outcome_panels:
    lines.append(r"\multicolumn{" + str(len(treatments_order) + 1) + r"}{l}{\textit{" + panel_label + r"}} \\[3pt]")
    
    # Replicated coefficients
    coef_cells = ["Replicated"]
    se_cells = [""]
    ref_cells = ["Published"]
    ref_se_cells = [""]
    n_cells = ["N"]
    r2_cells = ["$R^2$"]
    
    for t_var in treatments_order:
        key = (y_var, t_var)
        if key in results_lookup:
            r = results_lookup[key]
            # Stars for significance
            p = r["p_value"]
            stars = "^{***}" if p < 0.01 else "^{**}" if p < 0.05 else "^{*}" if p < 0.1 else ""
            coef_cells.append(f"${r['replicated_coef']:.3f}{stars}$")
            se_cells.append(f"$({r['replicated_se']:.3f})$")
            n_cells.append(f"${r['n_obs']}$")
            r2_cells.append(f"${r['r_squared']:.3f}$")
            
            if "published_coef" in r:
                ref_cells.append(f"${r['published_coef']:.3f}$")
                ref_se_cells.append(f"$({r['published_se']:.3f})$")
            else:
                ref_cells.append("---")
                ref_se_cells.append("")
        else:
            coef_cells.append("---")
            se_cells.append("")
            ref_cells.append("---")
            ref_se_cells.append("")
            n_cells.append("---")
            r2_cells.append("---")
    
    lines.append(" & ".join(coef_cells) + r" \\")
    lines.append(" & ".join(se_cells) + r" \\")
    lines.append(" & ".join(ref_cells) + r" \\")
    lines.append(" & ".join(ref_se_cells) + r" \\")
    lines.append(" & ".join(n_cells) + r" \\")
    lines.append(" & ".join(r2_cells) + r" \\[6pt]")

lines.append(r"\hline\hline")
lines.append(r"\multicolumn{" + str(len(treatments_order) + 1) + r"}{p{0.9\textwidth}}{\footnotesize Notes: OLS regressions with all 12 controls and HC1 robust standard errors in parentheses. Published values from Baiardi \& Naghi (2024) Table 1, Column 8 (available only for five-year effective rate). $^{***}p<0.01$, $^{**}p<0.05$, $^{*}p<0.1$.} \\")
lines.append(r"\end{tabular}")
lines.append(r"\end{table}")

tex_content = "\n".join(lines)
tex_path = TABLES_DIR / "table_replication.tex"
tex_path.write_text(tex_content)
print(f"=> {tex_path}")
print()
print(tex_content)

=> C:\Users\qgallea\Dropbox\work\claude_code\recast\papers\04_djankov_et_al_taxes\paper\tables\table_replication.tex

\begin{table}[htbp]
\centering
\caption{Replication of OLS Regressions --- All 12 Controls, HC1 Robust SE}
\label{tab:replication}
\small
\begin{tabular}{lccc}
\hline\hline
  & Statutory & Effective (1yr) & Effective (5yr) \\
\hline
\multicolumn{4}{l}{\textit{Panel A: Investment/GDP}} \\[3pt]
Replicated & $-0.064$ & $-0.117$ & $-0.189$ \\
 & $(0.121)$ & $(0.112)$ & $(0.116)$ \\
Published & --- & --- & $-0.189$ \\
 &  &  & $(0.118)$ \\
N & $61$ & $61$ & $61$ \\
$R^2$ & $0.472$ & $0.480$ & $0.494$ \\[6pt]
\multicolumn{4}{l}{\textit{Panel B: FDI/GDP}} \\[3pt]
Replicated & $-0.030$ & $-0.100^{*}$ & $-0.095$ \\
 & $(0.078)$ & $(0.060)$ & $(0.072)$ \\
Published & --- & --- & $-0.095$ \\
 &  &  & $(0.081)$ \\
N & $61$ & $61$ & $61$ \\
$R^2$ & $0.457$ & $0.477$ & $0.470$ \\[6pt]
\multicolumn{4}{l}{\textit{Panel C: Business Density}} \\[3pt]
Replicated & $-0.034$ & $-0.068$ & $-